# Train a single-language LID detector and compare it to off-the-shelf models

General-purpose language identifiers (langdetect, GlotLID, NLLB-LID) and LLMs are trained on high-resource languages and tend to **miss low-resource ones entirely** (see `compare-saved-runs.ipynb`).

This notebook shows that a local, single-purpose-built detector can beat off-the-shelf tools and frontier LLMs on a single low-resource language. It follows these steps:

1. take a **single-language dataset** (e.g. [the Şalom Ladino Corpus](https://mozilladatacollective.com/datasets/cmo1ks4zv004enr07la1rkr9x)),
2. turn it into a supervised problem by pairing its sentences (positives) with other-language text from Common Voice LID (negatives) — `train.build_training_data`,
3. train a specialist: **char n-gram logistic regression** (`train.train_logreg`, CPU-only) and/or **a fine-tuned HuggingFace LLM** (`train.finetune_llm`, e.g. Qwen-3.5b-0.6b),
4. compare every model on one shared question: *given a sentence, is it this language or not?*

Because the baselines predict among hundreds of languages while our detectors are binary, every model is reduced to the same yes/no decision so the scores are directly comparable.

## Setup

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

%matplotlib inline

# Load TOGETHER_API_KEY from the repo .env so the LLM baselines can authenticate.
try:
    from dotenv import load_dotenv
    load_dotenv(Path('../../../.env'))
except Exception as e:
    print('dotenv not loaded:', e)

from language_id.lang_codes_mapping import language_name, to_iso3
from language_id.models import TOGETHER_MODELS, get_model
from language_id.models.together import TogetherModel
from language_id.train import (
    DEFAULT_HF_MODEL_ID,
    build_training_data,
    evaluate_detector,
    finetune_llm,
    train_logreg,
)

## Configuration

- `TARGET_DATASET` / `TARGET_LANG`: the single-language dataset and the language (ISO-639-3) every row is in.
- `FINETUNE_HF` / `HF_MODEL_ID` / `HF_*`: fine-tune an off-the-shelf HF LLM as a second specialist. Point `HF_MODEL_ID` at any Hugging Face model (e.g. `Qwen/Qwen3-0.6B`) to swap the base model — nothing else changes.
- `STANDARD` / `LLMS` / `MAX_TEST_FOR_LLMS`: which baselines to compare against; when any LLMs are included, all models are scored on the same capped subset.

The number of Common Voice LID negatives isn't configured — `build_training_data` draws as many negatives as there are target-language positives.

In [2]:
TARGET_LANG = 'lad'  # Ladino (ISO-639-3)
TARGET_DATASET = 'alom-ladino-corpus-f409a2bb'

# Number of target-language *training* samples to use (negatives match it 1:1).
# Set to None to use every available positive; lower it to study how much data the
# specialists actually need. The test set is never capped, so scores stay comparable.
N_TRAIN_SAMPLES = 2

# Fine-tuned HF LLM specialist. Swap HF_MODEL_ID for any HF model id to try another base model.
FINETUNE_HF = True  # set False to skip the fine-tuning step that might require more computational resources
HF_MODEL_ID = DEFAULT_HF_MODEL_ID  # e.g. 'Qwen/Qwen3-0.6B'
HF_EPOCHS = 1
HF_BATCH_SIZE = 16

STANDARD = ['langdetect', 'glotlid', 'nllb-lid']
LLMS = ['gpt-oss-20b']           # [] to skip.
MAX_TEST_FOR_LLMS = 300          # cap shared test rows when LLMs are compared
MAX_OUTPUT_TOKENS = 256
SEED = 0

## 1. Build the training and test sets

`build_training_data` downloads the target corpus (a `.tar.gz` of `.txt` files), reads one sentence per line as positives, samples an equal number of other-language negatives from Common Voice LID, and splits both into train/test. Pass `n_train=N_TRAIN_SAMPLES` to cap the number of target-language training samples (negatives follow 1:1) — handy for studying how little data the specialists need. The test set is never capped, so scores stay comparable across training sizes. It returns:

- `train_df`: `[sentence, label]` — a binary problem (`TARGET_LANG` vs `other`).
- `test_df`: `[sentence, lang, is_target]` — keeps each row's true language so every model can be asked the same yes/no question.

In [3]:
train_df, test_df = build_training_data(TARGET_DATASET, TARGET_LANG, n_train=N_TRAIN_SAMPLES, seed=SEED)

n_pos = int((train_df['label'] == TARGET_LANG).sum())
print(f'train: {len(train_df)} rows ({n_pos} {TARGET_LANG} / {len(train_df) - n_pos} other)')
print(f'test:  {len(test_df)} rows '
      f'({int(test_df["is_target"].sum())} {TARGET_LANG} / {int((~test_df["is_target"]).sum())} other)')
train_df.head(3)

train: 4 rows (2 lad / 2 other)
test:  4276 rows (2138 lad / 2138 other)


,sentence,label
0,"Er schrieb auch Erzählungen und übersetzte ""„U...",other
1,Kwabena aware Canadani bea bi.,other
2,Esta todo lo mizmo,lad


## 2. Train the char n-gram logistic-regression specialist

A TF-IDF vectorizer over character n-grams feeds a logistic regression — the classic, strong, GPU-free LID feature. `train_logreg` returns a model implementing the project's `LIDModel` interface, so it's scored by the same code path as every other model.

In [4]:
logreg = train_logreg(train_df)
print('trained:', logreg.name)
print('char n-gram vocabulary size:', len(logreg.pipeline.named_steps['tfidf'].vocabulary_))
print('classes:', list(logreg.pipeline.named_steps['lr'].classes_))

trained: charngram-lr
char n-gram vocabulary size: 51
classes: ['lad', 'other']


## 3. (Optional) Fine-tune an off-the-shelf HF LLM as a second specialist

`finetune_llm` adds a binary classification head to any Hugging Face model (`HF_MODEL_ID`) and fine-tunes it on the same `train_df`. Swapping `HF_MODEL_ID` is the only change needed to try a different base model. Requires the optional `finetune` extra (`uv sync --extra finetune`); set `FINETUNE_HF = False` to skip.

In [5]:
hf_lid = None
if FINETUNE_HF:
    hf_lid = finetune_llm(
        train_df, TARGET_LANG,
        model_id=HF_MODEL_ID, epochs=HF_EPOCHS, batch_size=HF_BATCH_SIZE, seed=SEED,
    )
    print('fine-tuned model ready:', hf_lid.name)
else:
    print('FINETUNE_HF=False -> skipping HF fine-tuning')

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/1.50G [00:00<?, ?B/s]

Cancellation requested; stopping current tasks.


KeyboardInterrupt: 

## 4. Assemble the models to compare

The trained specialists plus the off-the-shelf baselines and (optionally) LLMs. When LLMs are included we score everyone on one stratified subset to keep the (slow, paid) API usage bounded.

In [ ]:
if LLMS and len(test_df) > MAX_TEST_FOR_LLMS:
    frac = MAX_TEST_FOR_LLMS / len(test_df)
    eval_df = test_df.groupby('is_target', group_keys=False).sample(frac=frac, random_state=SEED)
    print(f'LLMs included -> all models scored on a stratified {len(eval_df)}-row subset')
else:
    eval_df = test_df
    print(f'all models scored on the full {len(eval_df)}-row test set')

models = {logreg.name: logreg}
if hf_lid is not None:
    models[hf_lid.name] = hf_lid
for name in STANDARD:
    models[name] = get_model(name)
for name in LLMS:
    models[name] = TogetherModel(model_id=TOGETHER_MODELS[name], name=name, max_output_tokens=MAX_OUTPUT_TOKENS)
print('models:', list(models))

## 5. Score every model on the same question

`evaluate_detector` reduces each model's prediction to yes/no — *is this sentence `TARGET_LANG`?* — and scores it against the gold `is_target` column, so trained specialists and off-the-shelf tools are directly comparable.

In [ ]:
rows, failures = [], {}
for label, model in models.items():
    print(f'scoring {label} ...', flush=True)
    try:
        rows.append(evaluate_detector(model, eval_df, TARGET_LANG))
    except Exception as e:
        failures[label] = repr(e)
        print('  FAILED:', e)

if failures:
    print('skipped:', failures)

metric_cols = ['precision', 'recall', 'f1', 'accuracy']
results = pd.DataFrame(rows).sort_values('f1', ascending=False).reset_index(drop=True)
results.style.format({c: '{:.3f}' for c in metric_cols}).background_gradient(
    subset=metric_cols, cmap='Greens', vmin=0, vmax=1
)

### Detection scores by model

In [ ]:
ax = results.set_index('model')[['precision', 'recall', 'f1']].plot.bar(
    figsize=(max(7, 1.4 * len(results)), 4.5), rot=20, ylim=(0, 1)
)
ax.set_ylabel('score')
ax.set_title(f'Detecting {language_name(TARGET_LANG)} ({TARGET_LANG}): trained specialists vs. off-the-shelf')
ax.legend(title='metric', loc='lower right')
for container in ax.containers:
    ax.bar_label(container, fmt='%.2f', fontsize=6, padding=2)
plt.tight_layout()
plt.show()

## 6. Where a specialist wins

Target-language test sentences a trained specialist identifies correctly but a chosen baseline misses — the low-resource gap the specialist closes.

In [ ]:
SPECIALIST = logreg  # or hf_lid
BASELINE = 'glotlid'


def detect_target(model, texts):
    """Reduce a model's predictions to yes/no: is each sentence the target language?"""
    return [to_iso3(p.lang_code) == TARGET_LANG for p in model.predict_batch(texts)]


target_rows = eval_df[eval_df['is_target']]
spec_hit = pd.Series(detect_target(SPECIALIST, target_rows['sentence'].tolist()), index=target_rows.index)
base_hit = pd.Series(detect_target(models[BASELINE], target_rows['sentence'].tolist()), index=target_rows.index)
wins = target_rows[spec_hit & ~base_hit]
print(f'{len(wins)} / {len(target_rows)} {TARGET_LANG} sentences: {SPECIALIST.name} correct, {BASELINE} wrong')
wins[['sentence']].head(10)